<a href="https://colab.research.google.com/github/Sanjaysanjay31/24B11DS095_DriveReady_OPC_2026/blob/main/Day_3/Startup_Marketing_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 3 — Your First Agent: What an Agent Is, and LangChain

**Project:** AI Startup Marketing Agent  
**Curriculum:** Aditya University — One Person Company (12-Day Agentic AI Workshop)  
**Primary Provider:** OpenRouter — NVIDIA Nemotron 3.5 Super VL (Free)  
**Fallback Provider:** Groq — GPT-OSS 120B  

Day 3 transitions our startup marketing agent into a structured LangChain LCEL pipeline with conversation memory, automated evaluation testing via `tests.csv`, and agent task classification.

In [1]:
# Install LangChain packages and python-dotenv
!pip install -q langchain langchain-groq langchain-openrouter python-dotenv

import os

# Load environment variables from .env locally or Colab Secrets
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

try:
    from google.colab import userdata
    colab_openrouter = userdata.get("OPENROUTER_API_KEY")
    colab_groq = userdata.get("GROQ_API_KEY") or userdata.get("agentic_ai_GROQ_API_KEY")
except Exception:
    colab_openrouter = None
    colab_groq = None

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY") or colab_openrouter
GROQ_API_KEY = os.getenv("GROQ_API_KEY") or colab_groq

from langchain_groq import ChatGroq
from langchain_openrouter import ChatOpenRouter

# Primary: OpenRouter Nemotron 3.5 | Fallback: Groq GPT-OSS 120B
openrouter_llm = ChatOpenRouter(model="nvidia/nemotron-3.5-super-vl:free", temperature=0.3, api_key=OPENROUTER_API_KEY) if OPENROUTER_API_KEY else None
groq_llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0.3, api_key=GROQ_API_KEY) if GROQ_API_KEY else None

if openrouter_llm and groq_llm:
    llm = openrouter_llm.with_fallbacks([groq_llm])
else:
    llm = openrouter_llm or groq_llm

print("LangChain models initialized: OpenRouter (Primary) + Groq (Fallback)")

LangChain models initialized: OpenRouter (Primary) + Groq (Fallback)


In [2]:
# LangChain Expression Language (LCEL) chain with prompt template and memory placeholder
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

SYSTEM = """ROLE:   You are the AI Startup Marketing Agent for early-stage founders.
FACTS:  Viral Hook Rs 400. Short Script Rs 700. Storyboard Rs 1200. Voiceover Rs 500. Subtitles Rs 300. Video Assembly Rs 1500. SEO Pack Rs 350.
        We specialize in faceless 9:16 vertical videos for YouTube Shorts and Instagram Reels.
RULES:  Never promise or guarantee viral views or specific follower numbers. Quote prices strictly from FACTS.
        If a requested service is not listed in FACTS, say 'I will check with the team.'
FORMAT: Under 50 words, friendly, practical, and encouraging."""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),
    ("human", "{question}"),
])

chain = prompt | llm | StrOutputParser()
history = []

def ask(question: str) -> str:
    answer = chain.invoke({"history": history, "question": question})
    history.extend([("human", question), ("ai", answer)])
    return answer

print(ask("Hi, I'm Sanjay, founder of DevPulse. Do you make vertical videos for YouTube Shorts and Reels?"))
print(ask("What is my name and my startup?"))

Hello Sanjay! Yes, we specialize in high-retention 9:16 vertical videos for YouTube Shorts and Instagram Reels.
Your name is Sanjay, and you are the founder of DevPulse!


In [3]:
# Automated evaluation suite: tests.csv with required key phrase matching
import csv

test_cases = [
    {"question": "How much does viral hook generation cost?", "must_include": "400"},
    {"question": "Can you guarantee my Reel gets 1 million views?", "must_include": "never"},
    {"question": "What is the price of short script writing?", "must_include": "700"},
    {"question": "Do you create 9:16 vertical videos for Shorts?", "must_include": "9:16"},
    {"question": "Can you build a full iOS mobile app for me?", "must_include": "check with the team"}
]

with open("tests.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["question", "must_include"])
    writer.writeheader()
    writer.writerows(test_cases)

def run_tests(path: str = "tests.csv") -> None:
    rows = list(csv.DictReader(open(path, encoding="utf-8")))
    passed = 0
    print("=== Running Evaluation Tests ===")
    for row in rows:
        answer = chain.invoke({"history": [], "question": row["question"]})
        ok = row["must_include"].lower() in answer.lower()
        passed += ok
        status = "PASS" if ok else "FAIL"
        print(f"{status} | {row['question']} -> {answer[:65]}...")
    
    print(f"\nResult: {passed}/{len(rows)} passed ({passed / len(rows):.0%})")

run_tests()

=== Running Evaluation Tests ===
PASS | How much does viral hook generation cost? -> Viral Hook generation costs Rs 400...
PASS | Can you guarantee my Reel gets 1 million views? -> We never promise or guarantee viral views or follower numbers...
PASS | What is the price of short script writing? -> Short Script writing is Rs 700...
PASS | Do you create 9:16 vertical videos for Shorts? -> Yes, we specialize in faceless 9:16 vertical videos for YouTube Shorts...
PASS | Can you build a full iOS mobile app for me? -> That is not listed in our services; I will check with the team...

Result: 5/5 passed (100%)


In [4]:
# Day 3 Task: Agent or Not? (Classifying tasks into Plain Code, Single LLM Call, and Autonomous Agent)
marketing_jobs = [
    {
        "job": "Calculate package total from catalogue",
        "type": "Plain Code",
        "reason": "Deterministic price lookup and arithmetic; no reasoning or AI needed."
    },
    {
        "job": "Generate 3 viral hook variations from a problem statement",
        "type": "Single LLM Call",
        "reason": "Creative generation within a single prompt; no multi-step tool execution."
    },
    {
        "job": "Research competitor ads, write script, generate frames, and publish video",
        "type": "Autonomous Agent",
        "reason": "Multi-step goal requiring tool execution, observation, decision loop, and verification."
    }
]

print("=== Business Task Analysis: Plain Code vs LLM Call vs Agent ===\n")
for item in marketing_jobs:
    print(f"[{item['type']}] {item['job']}")
    print(f"   Reason: {item['reason']}\n")

=== Business Task Analysis: Plain Code vs LLM Call vs Agent ===

[Plain Code] Calculate package total from catalogue
   Reason: Deterministic price lookup and arithmetic; no reasoning or AI needed.

[Single LLM Call] Generate 3 viral hook variations from a problem statement
   Reason: Creative generation within a single prompt; no multi-step tool execution.

[Autonomous Agent] Research competitor ads, write script, generate frames, and publish video
   Reason: Multi-step goal requiring tool execution, observation, decision loop, and verification.

